# **CODE-5:**


In [48]:
'''
CODE-5: TO IMPUTE RH, PM VALUES AND IMPLEMENT RH-PM CORRECTION FACTOR
==========================================================================================
TO DO/RELOOK:
-> Recheck the final combined datamatrix making sure timestamp and place
        for ground and satellite are similar

INPUT           : chennai_final_combined_data
OUTPUT          : chennai_overall_datamatrix
==========================================================================================
'''

'\nCODE-5: TO IMPUTE RH, PM VALUES AND IMPLEMENT RH-PM CORRECTION FACTOR\n==========================================================================================\nTO DO/RELOOK:\n-> Recheck the final combined datamatrix making sure timestamp and place\n        for ground and satellite are similar\n\nINPUT           : chennai_final_combined_data\nOUTPUT          : chennai_overall_datamatrix\n==========================================================================================\n'

In [49]:
pip install datawig

     |████████████████████████████████| 13.8MB 317kB/s 
ERROR: xarray 0.15.1 has requirement numpy>=1.15, but you'll have numpy 1.14.6 which is incompatible.
ERROR: umap-learn 0.4.3 has requirement numpy>=1.17, but you'll have numpy 1.14.6 which is incompatible.
ERROR: tifffile 2020.5.30 has requirement numpy>=1.15.1, but you'll have numpy 1.14.6 which is incompatible.
ERROR: tensorflow 2.2.0 has requirement numpy<2.0,>=1.16.0, but you'll have numpy 1.14.6 which is incompatible.
ERROR: spacy 2.2.4 has requirement numpy>=1.15.0, but you'll have numpy 1.14.6 which is incompatible.
ERROR: plotnine 0.6.0 has requirement numpy>=1.16.0, but you'll have numpy 1.14.6 which is incompatible.
ERROR: numba 0.48.0 has requirement numpy>=1.15, but you'll have numpy 1.14.6 which is incompatible.
ERROR: imgaug 0.2.9 has requirement numpy>=1.15.0, but you'll have numpy 1.14.6 which is incompatible.
ERROR: google-colab 1.0.0 has requirement pandas~=1.0.0; python_version >= "3.0", but you'll have pandas 

In [0]:
import pandas as pd
import datawig  #for imputation
import numpy as np

In [0]:
chennai_combined_data = pd.read_csv("chennai_final_combined_data7.csv")

In [52]:
chennai_combined_data

,Unnamed: 0,PM2.5,Temp,RH,place,From Date,Unnamed: 0.1,Year,Month,Hour,Day,RoundedMinute,Latitude,Longitude,Place,time_stamp,updated_AOD_Land_and_Ocean_datawig,Optical_Depth_Land_And_Ocean,Image_Optical_Depth_Land_And_Ocean,Land_sea_Flag,Land_Ocean_Quality_Flag
0,0,0.00,0.00,0.00,Manali,2016-01-01 13:45:00,0,2016.0,1.0,13.0,1.0,45,13.146203,80.274750,Manali,2016-01-01 13:55:42,0.448998,-9999.000,NaN,1.0,-9999.0
1,1,0.00,0.00,0.00,Manali,2016-01-01 14:00:00,1,2016.0,1.0,13.0,1.0,45,13.157624,80.266487,Manali,2016-01-01 13:55:42,0.443463,-9999.000,NaN,1.0,-9999.0
2,2,0.00,0.00,0.00,Velachery,2016-01-01 13:45:00,2,2016.0,1.0,13.0,1.0,45,12.990589,80.211304,Velachery,2016-01-01 13:55:39,0.309563,-9999.000,NaN,1.0,-9999.0
3,3,34.27,30.29,73.45,Alandur,2016-01-02 13:00:00,3,2016.0,1.0,13.0,2.0,0,13.003789,80.203102,Alandur,2016-01-02 13:00:37,0.283564,-9999.000,NaN,1.0,-9999.0
4,4,29.32,33.59,60.16,Manali,2016-01-02 13:00:00,4,2016.0,1.0,13.0,2.0,0,13.162173,80.271301,Manali,2016-01-02 13:00:39,0.450397,-9999.000,NaN,1.0,-9999.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1029,205,7.34,0.00,54.75,Velachery,2020-03-29 13:30:00,1029,2020.0,3.0,13.0,29.0,15,12.987316,80.204910,Velachery,2020-03-29 13:25:23,0.277000,0.277,0.277,1.0,3.0
1030,206,38.72,0.00,50.16,Manali,2020-03-30 14:00:00,1030,2020.0,3.0,14.0,30.0,0,13.144199,80.260208,Manali,2020-03-30 14:08:17,0.519539,-9999.000,NaN,1.0,-9999.0
1031,207,14.41,0.00,49.76,Velachery,2020-03-30 14:00:00,1031,2020.0,3.0,14.0,30.0,0,12.986140,80.214638,Velachery,2020-03-30 14:08:16,0.607980,-9999.000,NaN,1.0,-9999.0
1032,208,23.86,0.00,49.39,Velachery,2020-03-30 14:15:00,1032,2020.0,3.0,14.0,30.0,0,12.974220,80.210815,Velachery,2020-03-30 14:08:16,0.619227,-9999.000,NaN,1.0,-9999.0


In [0]:
#PM IMPUTATION
#1. Dividing into year-wise data for imputation
def year_wise(str, year):
     return chennai_combined_data[chennai_combined_data[str] == year]  
yr_2016 = year_wise('Year', 2016)
yr_2017 = year_wise('Year', 2017)
yr_2018 = year_wise('Year', 2018)
yr_2019 = year_wise('Year', 2019)
yr_2020 = year_wise('Year', 2020)

In [54]:
chennai_combined_data.groupby(['Year']).size().reset_index(name='count')

,Year,count
0,2016.0,232
1,2017.0,208
2,2018.0,198
3,2019.0,186
4,2020.0,210


In [0]:
# IMPUTATION
#df - yearwise dataframe to be imputed, str - column name
#month_num - month number, null_value - null value that is to be refilled 
#Imputation is done for every month, in a certain place, based on weights(distance) of latitude and longitude 
#Haversine formula is not included to make things simpler

In [0]:
def imputation(df, str1, str2, str3,):
    
    def month_wise_df(df, month_num, str1, str2, str3, null_value):
        df_mon = df[df['Month'] == month_num]
        
        x_columns = ['Latitude', 'Longitude','Place']
        
        value = df_mon[df_mon[str1] != null_value]
        no_value = df_mon[df_mon[str1] == null_value]
        
        imputer = datawig.SimpleImputer(
            input_columns = x_columns, # column(s) containing information about the column we want to impute
            output_column = str1, # the column we'd like to impute values for
            output_path = 'datawig_imputation' # stores model data and metrics
            )
        imputer.fit(train_df = value, num_epochs = 120)
        imputed = imputer.predict(no_value)
        imputed.head()
        
        datawig_imputation_data = list()
        predictions = np.asarray(imputed[str2])
        cnt = 0
        for i in range(df_mon.shape[0]):
            if(df_mon.iloc[i][str1] == null_value):
                datawig_imputation_data.append(predictions[cnt])
                cnt+=1
            else:
                datawig_imputation_data.append(df_mon.iloc[i][str1])
        print(cnt,"rows have been imputed")
        df_mon[str3] = np.asarray(datawig_imputation_data)
        
        return df_mon
        
    mon_1 = month_wise_df(df, 1, str1, str2, str3, 0)
    mon_2 = month_wise_df(df, 2, str1, str2, str3, 0)
    mon_3 = month_wise_df(df, 3, str1, str2, str3, 0)
    
    month_dflist = [mon_1, mon_2, mon_3]
    imputed_monthwise_df = pd.DataFrame()
    for mon in month_dflist:
        imputed_monthwise_df = imputed_monthwise_df.append(mon)
        
    return imputed_monthwise_df

In [60]:
#calling Imputation for filling PM values
yr_2016_PM_imputed = imputation(yr_2016,'PM2.5','PM2.5_imputed','PM_updated')
yr_2017_PM_imputed = imputation(yr_2017,'PM2.5','PM2.5_imputed','PM_updated')
yr_2018_PM_imputed = imputation(yr_2018,'PM2.5','PM2.5_imputed','PM_updated')
yr_2019_PM_imputed = imputation(yr_2019,'PM2.5','PM2.5_imputed','PM_updated')
yr_2020_PM_imputed = imputation(yr_2020,'PM2.5','PM2.5_imputed','PM_updated')

2020-06-09 14:58:53,807 [INFO]  
========== start: fit model
2020-06-09 14:58:53,811 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:58:53,867 [INFO]  Epoch[0] Batch [0-2]	Speed: 1002.25 samples/sec	cross-entropy=11.345235	PM2.5-accuracy=0.000000
2020-06-09 14:58:53,887 [INFO]  Epoch[0] Train-cross-entropy=16.210867
2020-06-09 14:58:53,891 [INFO]  Epoch[0] Train-PM2.5-accuracy=0.000000
2020-06-09 14:58:53,893 [INFO]  Epoch[0] Time cost=0.078
2020-06-09 14:58:53,899 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:58:53,917 [INFO]  Epoch[0] Validation-cross-entropy=8.075699
2020-06-09 14:58:53,922 [INFO]  Epoch[0] Validation-PM2.5-accuracy=0.000000
2020-06-09 14:58:53,974 [INFO]  Epoch[1] Batch [0-2]	Speed: 1024.37 samples/sec	cross-entropy=10.652667	PM2.5-accuracy=0.000000
2020-06-09 14:58:53,995 [INFO]  Epoch[1] Train-cross-entropy=15.675165
2020-06-09 14:58:54,000 [INFO]  Epoch[1] Train-PM2.5-accuracy=0.000000
2020-06-09 14:58:54,004 [INFO]  Ep

14 rows have been imputed


2020-06-09 14:58:55,593 [INFO]  
========== start: fit model
2020-06-09 14:58:55,597 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:58:55,654 [INFO]  Epoch[0] Batch [0-2]	Speed: 1003.87 samples/sec	cross-entropy=18.352882	PM2.5-accuracy=0.000000
2020-06-09 14:58:55,674 [INFO]  Epoch[0] Train-cross-entropy=14.501904
2020-06-09 14:58:55,678 [INFO]  Epoch[0] Train-PM2.5-accuracy=0.000000
2020-06-09 14:58:55,681 [INFO]  Epoch[0] Time cost=0.078
2020-06-09 14:58:55,687 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:58:55,704 [INFO]  Epoch[0] Validation-cross-entropy=5.456439
2020-06-09 14:58:55,707 [INFO]  Epoch[0] Validation-PM2.5-accuracy=0.000000
2020-06-09 14:58:55,759 [INFO]  Epoch[1] Batch [0-2]	Speed: 1021.12 samples/sec	cross-entropy=17.559775	PM2.5-accuracy=0.000000
2020-06-09 14:58:55,780 [INFO]  Epoch[1] Train-cross-entropy=13.793141
2020-06-09 14:58:55,785 [INFO]  Epoch[1] Train-PM2.5-accuracy=0.000000
2020-06-09 14:58:55,789 [INFO]  Ep

11 rows have been imputed


2020-06-09 14:58:56,614 [INFO]  
========== start: fit model
2020-06-09 14:58:56,619 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:58:56,675 [INFO]  Epoch[0] Batch [0-2]	Speed: 1008.19 samples/sec	cross-entropy=15.786303	PM2.5-accuracy=0.000000
2020-06-09 14:58:56,697 [INFO]  Epoch[0] Train-cross-entropy=15.189493
2020-06-09 14:58:56,702 [INFO]  Epoch[0] Train-PM2.5-accuracy=0.000000
2020-06-09 14:58:56,707 [INFO]  Epoch[0] Time cost=0.082
2020-06-09 14:58:56,715 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:58:56,734 [INFO]  Epoch[0] Validation-cross-entropy=11.419115
2020-06-09 14:58:56,739 [INFO]  Epoch[0] Validation-PM2.5-accuracy=0.000000
2020-06-09 14:58:56,793 [INFO]  Epoch[1] Batch [0-2]	Speed: 990.92 samples/sec	cross-entropy=12.677509	PM2.5-accuracy=0.000000
2020-06-09 14:58:56,816 [INFO]  Epoch[1] Train-cross-entropy=13.420330
2020-06-09 14:58:56,822 [INFO]  Epoch[1] Train-PM2.5-accuracy=0.000000
2020-06-09 14:58:56,827 [INFO]  Ep

...

2020-06-09 14:59:08,156 [INFO]  
========== start: fit model
2020-06-09 14:59:08,161 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:59:08,215 [INFO]  Epoch[0] Batch [0-2]	Speed: 1012.28 samples/sec	cross-entropy=10.764598	PM2.5-accuracy=0.000000
2020-06-09 14:59:08,219 [INFO]  Epoch[0] Train-cross-entropy=10.764598
2020-06-09 14:59:08,224 [INFO]  Epoch[0] Train-PM2.5-accuracy=0.000000
2020-06-09 14:59:08,228 [INFO]  Epoch[0] Time cost=0.061
2020-06-09 14:59:08,234 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:59:08,253 [INFO]  Epoch[0] Validation-cross-entropy=2.273558
2020-06-09 14:59:08,256 [INFO]  Epoch[0] Validation-PM2.5-accuracy=0.000000
2020-06-09 14:59:08,308 [INFO]  Epoch[1] Batch [0-2]	Speed: 1023.88 samples/sec	cross-entropy=5.878419	PM2.5-accuracy=0.000000
2020-06-09 14:59:08,313 [INFO]  Epoch[1] Train-cross-entropy=5.878419
2020-06-09 14:59:08,317 [INFO]  Epoch[1] Train-PM2.5-accuracy=0.000000
2020-06-09 14:59:08,322 [INFO]  Epoc

30 rows have been imputed


2020-06-09 14:59:09,657 [INFO]  
========== start: fit model
2020-06-09 14:59:09,663 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:59:09,718 [INFO]  Epoch[0] Batch [0-2]	Speed: 1006.64 samples/sec	cross-entropy=12.694298	PM2.5-accuracy=0.000000
2020-06-09 14:59:09,739 [INFO]  Epoch[0] Train-cross-entropy=11.142404
2020-06-09 14:59:09,744 [INFO]  Epoch[0] Train-PM2.5-accuracy=0.000000
2020-06-09 14:59:09,749 [INFO]  Epoch[0] Time cost=0.080
2020-06-09 14:59:09,755 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:59:09,773 [INFO]  Epoch[0] Validation-cross-entropy=6.505476
2020-06-09 14:59:09,777 [INFO]  Epoch[0] Validation-PM2.5-accuracy=0.000000
2020-06-09 14:59:09,829 [INFO]  Epoch[1] Batch [0-2]	Speed: 1020.00 samples/sec	cross-entropy=8.627203	PM2.5-accuracy=0.000000
2020-06-09 14:59:09,850 [INFO]  Epoch[1] Train-cross-entropy=7.001240
2020-06-09 14:59:09,854 [INFO]  Epoch[1] Train-PM2.5-accuracy=0.000000
2020-06-09 14:59:09,859 [INFO]  Epoc

16 rows have been imputed


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [61]:
#calling Imputation for filling RH values
yr_2016_overall = imputation(yr_2016_PM_imputed, 'RH', 'RH_imputed', 'RH_updated')
yr_2017_overall = imputation(yr_2017_PM_imputed, 'RH', 'RH_imputed', 'RH_updated')
yr_2018_overall = imputation(yr_2018_PM_imputed, 'RH', 'RH_imputed', 'RH_updated')
yr_2019_overall = imputation(yr_2019_PM_imputed, 'RH', 'RH_imputed', 'RH_updated')
yr_2020_overall = imputation(yr_2020_PM_imputed, 'RH', 'RH_imputed', 'RH_updated')

2020-06-09 14:59:10,768 [INFO]  
========== start: fit model
2020-06-09 14:59:10,772 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:59:10,828 [INFO]  Epoch[0] Batch [0-2]	Speed: 987.32 samples/sec	cross-entropy=17.469763	RH-accuracy=0.000000
2020-06-09 14:59:10,849 [INFO]  Epoch[0] Train-cross-entropy=15.322024
2020-06-09 14:59:10,853 [INFO]  Epoch[0] Train-RH-accuracy=0.000000
2020-06-09 14:59:10,857 [INFO]  Epoch[0] Time cost=0.079
2020-06-09 14:59:10,863 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:59:10,883 [INFO]  Epoch[0] Validation-cross-entropy=24.255781
2020-06-09 14:59:10,887 [INFO]  Epoch[0] Validation-RH-accuracy=0.000000
2020-06-09 14:59:10,940 [INFO]  Epoch[1] Batch [0-2]	Speed: 996.78 samples/sec	cross-entropy=15.880390	RH-accuracy=0.000000
2020-06-09 14:59:10,961 [INFO]  Epoch[1] Train-cross-entropy=13.960451
2020-06-09 14:59:10,965 [INFO]  Epoch[1] Train-RH-accuracy=0.000000
2020-06-09 14:59:10,969 [INFO]  Epoch[1] Time cost

19 rows have been imputed


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
2020-06-09 14:59:11,859 [INFO]  
========== start: fit model
2020-06-09 14:59:11,862 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:59:11,919 [INFO]  Epoch[0] Batch [0-2]	Speed: 1020.77 samples/sec	cross-entropy=16.617933	RH-accuracy=0.000000
2020-06-09 14:59:11,938 [INFO]  Epoch[0] Train-cross-entropy=16.185444
2020-06-09 14:59:11,942 [INFO]  Epoch[0] Train-RH-accuracy=0.000000
2020-06-09 14:59:11,946 [INFO]  Epoch[0] Time cost=0.077
2020-06-09 14:59:11,952 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:59:11,969 [INFO]  Epoch[0] Validation-cross-entropy=2.763333
2020-06-09 14:59:11,973 [INFO]  Ep

11 rows have been imputed


2020-06-09 14:59:12,991 [INFO]  
========== start: fit model
2020-06-09 14:59:12,996 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:59:13,056 [INFO]  Epoch[0] Batch [0-2]	Speed: 888.17 samples/sec	cross-entropy=15.236794	RH-accuracy=0.000000
2020-06-09 14:59:13,077 [INFO]  Epoch[0] Train-cross-entropy=15.980288
2020-06-09 14:59:13,081 [INFO]  Epoch[0] Train-RH-accuracy=0.000000
2020-06-09 14:59:13,085 [INFO]  Epoch[0] Time cost=0.083
2020-06-09 14:59:13,091 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:59:13,110 [INFO]  Epoch[0] Validation-cross-entropy=6.754843
2020-06-09 14:59:13,114 [INFO]  Epoch[0] Validation-RH-accuracy=0.000000
2020-06-09 14:59:13,165 [INFO]  Epoch[1] Batch [0-2]	Speed: 1014.41 samples/sec	cross-entropy=15.587337	RH-accuracy=0.000000
2020-06-09 14:59:13,188 [INFO]  Epoch[1] Train-cross-entropy=15.318995
2020-06-09 14:59:13,192 [INFO]  Epoch[1] Train-RH-accuracy=0.000000
2020-06-09 14:59:13,198 [INFO]  Epoch[1] Time cost

...

32 rows have been imputed


2020-06-09 14:59:37,006 [INFO]  
========== start: fit model
2020-06-09 14:59:37,011 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:59:37,052 [INFO]  Epoch[0] Batch [0-1]	Speed: 1011.24 samples/sec	cross-entropy=16.811348	RH-accuracy=0.000000
2020-06-09 14:59:37,057 [INFO]  Epoch[0] Train-cross-entropy=16.811348
2020-06-09 14:59:37,061 [INFO]  Epoch[0] Train-RH-accuracy=0.000000
2020-06-09 14:59:37,065 [INFO]  Epoch[0] Time cost=0.049
2020-06-09 14:59:37,071 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:59:37,089 [INFO]  Epoch[0] Validation-cross-entropy=50.148392
2020-06-09 14:59:37,093 [INFO]  Epoch[0] Validation-RH-accuracy=0.000000
2020-06-09 14:59:37,128 [INFO]  Epoch[1] Batch [0-1]	Speed: 1033.16 samples/sec	cross-entropy=16.168075	RH-accuracy=0.000000
2020-06-09 14:59:37,133 [INFO]  Epoch[1] Train-cross-entropy=16.168075
2020-06-09 14:59:37,137 [INFO]  Epoch[1] Train-RH-accuracy=0.000000
2020-06-09 14:59:37,140 [INFO]  Epoch[1] Time co

32 rows have been imputed


2020-06-09 14:59:38,352 [INFO]  
========== start: fit model
2020-06-09 14:59:38,356 [WARNING]  Already bound, ignoring bind()
2020-06-09 14:59:38,420 [INFO]  Epoch[0] Batch [0-2]	Speed: 1001.33 samples/sec	cross-entropy=16.361950	RH-accuracy=0.000000
2020-06-09 14:59:38,425 [INFO]  Epoch[0] Train-cross-entropy=16.361950
2020-06-09 14:59:38,430 [INFO]  Epoch[0] Train-RH-accuracy=0.000000
2020-06-09 14:59:38,434 [INFO]  Epoch[0] Time cost=0.068
2020-06-09 14:59:38,441 [INFO]  Saved checkpoint to "datawig_imputation/model-0000.params"
2020-06-09 14:59:38,460 [INFO]  Epoch[0] Validation-cross-entropy=15.880401
2020-06-09 14:59:38,465 [INFO]  Epoch[0] Validation-RH-accuracy=0.000000
2020-06-09 14:59:38,518 [INFO]  Epoch[1] Batch [0-2]	Speed: 990.86 samples/sec	cross-entropy=15.589709	RH-accuracy=0.000000
2020-06-09 14:59:38,523 [INFO]  Epoch[1] Train-cross-entropy=15.589709
2020-06-09 14:59:38,528 [INFO]  Epoch[1] Train-RH-accuracy=0.000000
2020-06-09 14:59:38,532 [INFO]  Epoch[1] Time cos

25 rows have been imputed


In [0]:
# 3.Appending all years back
imputed_year_dflist = [yr_2016_overall, yr_2017_overall, yr_2018_overall,
                       yr_2019_overall, yr_2020_overall]
chennai_combined_data2 = pd.DataFrame()


In [0]:
for imputed_df in imputed_year_dflist:
    chennai_combined_data2 = chennai_combined_data2.append(imputed_df)

In [68]:
chennai_combined_data2['PM_updated'].mean(axis = 0, skipna = True) 

45.434255911546096

In [69]:
chennai_combined_data2['PM2.5'].replace(0, np.NaN, inplace = True)
chennai_combined_data2['PM2.5'].mean(axis = 0, skipna = True)

48.38841095890415

In [70]:
chennai_combined_data2['RH_updated'].mean(axis = 0, skipna = True)

58.11690366931941

In [71]:
chennai_combined_data2['RH'].replace(0, np.NaN, inplace = True)
chennai_combined_data2['RH'].mean(axis = 0, skipna = True)

54.60396551724141

In [0]:
#Corrected PM and corrected RH

In [0]:
chennai_combined_data2.to_csv('chennai_overall_data.csv')